# Women in STEM Graduates: A Data Storytelling Project

This project explores how the share of women graduating in STEM fields has changed over time across countries.

The goal is to identify long-term trends, compare selected countries, and highlight how close different countries are to gender parity in STEM education.

The analysis combines:
- time series comparison
- animated visual storytelling
- global mapping

This notebook was created as part of a data storytelling project.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

## 1. Load the data

We begin by loading the dataset containing the share of female graduates in STEM fields by country and year.

In [10]:
# Load dataset
FILE_PATH = "../data/share-graduates-stem-female.csv"
df = pd.read_csv(FILE_PATH)

# Preview the data
df.head()

,Entity,Code,Year,"Female share of graduates from Science, Technology, Engineering and Mathematics (STEM) programmes, tertiary (%)"
0,Albania,ALB,2000,32.43243
1,Albania,ALB,2003,44.20063
2,Albania,ALB,2011,48.06421
3,Albania,ALB,2015,52.78050
4,Albania,ALB,2016,48.29606


## 2. Inspect and prepare the dataset

Before creating visualizations, we inspect the structure of the dataset and clean the most important fields.

In [11]:
# Check dataset structure
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nMissing values:")
print(df.isna().sum())

Shape: (1223, 4)

Columns:
Index(['Entity', 'Code', 'Year',
       'Female share of graduates from Science, Technology, Engineering and Mathematics (STEM) programmes, tertiary (%)'],
      dtype='object')

Missing values:
Entity                                                                                                             0
Code                                                                                                               0
Year                                                                                                               0
Female share of graduates from Science, Technology, Engineering and Mathematics (STEM) programmes, tertiary (%)    0
dtype: int64


In [12]:
# Rename columns properly
df = df.rename(columns={
    "Entity": "country",
    "Year": "year",
    "Female share of graduates from Science, Technology, Engineering and Mathematics (STEM) programmes, tertiary (%)": "female_stem_share"
})

# Ensure correct types
df["year"] = df["year"].astype(int)
df["female_stem_share"] = pd.to_numeric(df["female_stem_share"], errors="coerce")

# Drop missing values (after conversion)
df = df.dropna(subset=["country", "year", "female_stem_share"])

# Sort for clean analysis and plotting
df = df.sort_values(["country", "year"]).reset_index(drop=True)

## 3. General overview

This section provides a quick overview of the time coverage and countries included in the dataset.

In [13]:
# Basic overview
print("Number of countries:", df["country"].nunique())
print("Year range:", df["year"].min(), "to", df["year"].max())

Number of countries: 155
Year range: 1998 to 2019


## 4. Interactive global view

We first create a global animated map to show how the share of female STEM graduates changes across countries over time.

This helps identify broad regional patterns and long-term shifts.

In [98]:
# Prepare data specifically for the animated map
df_map = df.copy()

# Keep only needed columns
df_map = df_map[["country", "year", "female_stem_share"]].copy()

# Make sure year is numeric, then sort chronologically
df_map["year"] = pd.to_numeric(df_map["year"], errors="coerce")
df_map = df_map.dropna(subset=["year", "female_stem_share"])
df_map["year"] = df_map["year"].astype(int)

# Sort rows so animation frames are built in the right order
df_map = df_map.sort_values(["year", "country"]).reset_index(drop=True)

# Convert year to string AFTER sorting
df_map["year_str"] = df_map["year"].astype(str)

fig_map = px.choropleth(
    df_map,
    locations="country",
    locationmode="country names",
    color="female_stem_share",
    hover_name="country",
    animation_frame="year_str",
    animation_group="country",
    color_continuous_scale="Viridis",
    range_color=[df_map["female_stem_share"].min(), df_map["female_stem_share"].max()],
    labels={"female_stem_share": "Female STEM share (%)"}
)

fig_map.update_layout(
    template="plotly_white",

    height=800,

    margin=dict(l=20, r=80, t=100, b=40),

    coloraxis_colorbar=dict(
        title="Female STEM share (%)",
        x=1.02
    ),

    geo=dict(
        domain=dict(x=[0.02, 0.9], y=[0.25, 0.95])
    ),

    annotations=[
        dict(
            text="Global Share of Female STEM Graduates Over Time",
            x=0.46,
            y=1.12,
            xref="paper",
            yref="paper",
            showarrow=False,
            xanchor="center",
            font=dict(size=24)
        )
    ]
)
fig_map.show()

## Note on data availability

As the animation shows, some countries appear and disappear over time.

This happens because not every country has data available for every year in the dataset. Missing observations in specific years cause countries to temporarily drop out of the animated map.

To make a fair comparison, we now identify the countries that have values for every single year in the selected time period and create a bar chart based only on those complete cases.

In [16]:
# Identify the full year range in the dataset
all_years = sorted(df["year"].unique())
num_years = len(all_years)

print("Years in dataset:", all_years)
print("Total number of years:", num_years)

Years in dataset: [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Total number of years: 22


In [17]:
# Count how many unique years are available for each country
country_year_counts = (
    df.groupby("country")["year"]
    .nunique()
    .reset_index(name="n_years")
)

# Keep only countries that have data for every year
complete_countries = country_year_counts[country_year_counts["n_years"] == num_years]["country"]

print("Number of countries with complete data:", len(complete_countries))
print(sorted(complete_countries.tolist()))

Number of countries with complete data: 0
[]


In [44]:
# Choose a common comparison window
start_year_complete = 2002
end_year_complete = 2015

df_common = df[
    (df["year"] >= start_year_complete) &
    (df["year"] <= end_year_complete)
].copy()

all_years_common = sorted(df_common["year"].unique())
num_years_common = len(all_years_common)

print("Years in common window:", all_years_common)
print("Total years in common window:", num_years_common)

Years in common window: [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]
Total years in common window: 14


In [45]:
# Count available years for each country within the selected window
country_year_counts_common = (
    df_common.groupby("country")["year"]
    .nunique()
    .reset_index(name="n_years")
)

# Keep only countries with complete coverage in this window
complete_countries_common = country_year_counts_common[
    country_year_counts_common["n_years"] == num_years_common
]["country"]

print("Number of countries with complete data in selected period:", len(complete_countries_common))
print(sorted(complete_countries_common.tolist()))

Number of countries with complete data in selected period: 10
['Denmark', 'Estonia', 'Latvia', 'Lithuania', 'North Macedonia', 'Portugal', 'Slovakia', 'Slovenia', 'Spain', 'United Kingdom']


## Selecting countries with complete data

When using the full dataset, no country had observations for every single year, which made direct comparisons unreliable.

To address this, we restricted the analysis to a common time window (2002–2015). Within this period, we first identified countries with complete data for every year.

From this consistent group, we then selected a smaller subset of countries for clearer visualization and comparison:

- Denmark
- Estonia
- Latvia
- Lithuania
- Portugal
- Spain
- United Kingdom

Focusing on these countries ensures that all time series are complete and comparable
trends are not distorted by missing data
the visualization remains readable and interpretable

## Trends over time for countries with complete data

We now visualize how the share of female STEM graduates evolves over time for countries with complete data coverage.

Using this filtered set ensures that all trends are directly comparable, without interruptions caused by missing observations.

In [64]:
# Use only complete countries
selected_countries_focus = [
    "Denmark",
    "United Kingdom",
    "Estonia",
    "Spain",
    "Portugal",
    "Latvia",
    "Lithuania"
]

df_line_complete = df_common[
    df_common["country"].isin(selected_countries_focus)
].copy()

# Years for animation
years_line = list(range(start_year_complete, end_year_complete + 1))

# Helper function (same logic as before)
def data_until_year_complete(country, year):
    return df_line_complete[
        (df_line_complete["country"] == country) &
        (df_line_complete["year"] <= year)
    ].sort_values("year")

In [70]:
fig_line_complete = go.Figure()

initial_year = years_line[0]

# Initial lines
for country in selected_countries_focus:
    d = data_until_year_complete(country, initial_year)

    fig_line_complete.add_trace(
        go.Scatter(
            x=d["year"],
            y=d["female_stem_share"],
            mode="lines",
            line=dict(width=4),
            name=country,
            showlegend=False,
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Year: %{x}<br>"
                "Female share: %{y:.1f}%<extra></extra>"
            )
        )
    )

# Parity line
fig_line_complete.add_hline(
    y=50,
    line_dash="dash",
    line_color="black",
    opacity=0.5
)

# Static legend (same trick as you used)
for country in selected_countries_focus:
    fig_line_complete.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="lines",
            line=dict(width=4),
            name=country,
            showlegend=True
        )
    )

In [71]:
frames = []

for year in years_line:
    frame_data = []

    for country in selected_countries_focus:
        d = data_until_year_complete(country, year)

        frame_data.append(
            go.Scatter(
                x=d["year"],
                y=d["female_stem_share"],
                mode="lines",
                line=dict(width=4),
                name=country,
                showlegend=False,
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Year: %{x}<br>"
                    "Female share: %{y:.1f}%<extra></extra>"
                )
            )
        )

    frames.append(go.Frame(data=frame_data, name=str(year)))

fig_line_complete.frames = frames

In [73]:
fig_line_complete.update_layout(
    title=f"Evolution of Female STEM Graduates ({start_year_complete}–{end_year_complete})",
    xaxis_title="Year",
    title_x=0.5,
    yaxis_title="Female STEM share (%)",
    template="plotly_white",
    hovermode="x unified",
    height=650,
    width=1400,

    xaxis=dict(
        range=[start_year_complete, end_year_complete],
        tickmode="linear",
        dtick=1
    ),

    yaxis=dict(
        range=[24, 50]
    ),

    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=1.02
    ),

    margin=dict(l=80, r=220, t=80, b=100),

    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "x": -0.08,
            "y": 1.0,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 500, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 300}
                        }
                    ]
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }
                    ]
                }
            ]
        }
    ],

    sliders=[
        {
            "currentvalue": {
                "prefix": "Year: ",
                "font": {"size": 16}
            },
            "pad": {"t": 40},
            "steps": [
                {
                    "label": str(year),
                    "method": "animate",
                    "args": [
                        [str(year)],
                        {
                            "frame": {"duration": 300, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 200}
                        }
                    ]
                }
                for year in years_line
            ]
        }
    ]
)

fig_line_complete.show()

This focused comparison reveals clear differences in the representation of women in STEM across countries. Estonia and Latvia consistently maintain higher shares, indicating stronger female participation, while countries such as Denmark and Spain remain comparatively lower throughout most of the period. Notably, Portugal demonstrates a late upward trend, suggesting recent progress. Overall, these patterns highlight both persistent gaps and signs of gradual improvement across Europe.